In [1]:
# ====================================================
# 📦 IMPORT LIBRARIES FOR DATA CLEANING
# ====================================================

import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

print("✅ Libraries imported successfully!")
print(f"📊 Pandas version: {pd.__version__}")

✅ Libraries imported successfully!
📊 Pandas version: 2.3.2


In [2]:
# ====================================================
# 📂 SETUP FILE PATHS
# ====================================================

RAW_PATH = Path("../dataset/raw/")
PROCESSED_PATH = Path("../dataset/processed/")

# Create processed folder if not exists
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

print(f"📥 Raw data path:       {RAW_PATH.absolute()}")
print(f"📤 Processed data path: {PROCESSED_PATH.absolute()}")

📥 Raw data path:       e:\Clash Royale\notebooks\..\dataset\raw
📤 Processed data path: e:\Clash Royale\notebooks\..\dataset\processed


In [3]:
# ====================================================
# 📂 LOAD RAW DATASETS
# ====================================================

cards_df = pd.read_csv(RAW_PATH / "cards.csv")
decks_df = pd.read_csv(RAW_PATH / "decks.csv")
battles_df = pd.read_csv(RAW_PATH / "battles.csv")
meta_df = pd.read_csv(RAW_PATH / "meta_stats.csv")

print("✅ All raw datasets loaded!\n")
print(f"📦 Cards:    {cards_df.shape}")
print(f"🃏 Decks:    {decks_df.shape}")
print(f"⚔️  Battles:  {battles_df.shape}")
print(f"📊 Meta:     {meta_df.shape}")

✅ All raw datasets loaded!

📦 Cards:    (98, 12)
🃏 Decks:    (500, 14)
⚔️  Battles:  (5000, 12)
📊 Meta:     (490, 6)


In [4]:
# ====================================================
# 🔍 MISSING VALUES - BEFORE CLEANING
# ====================================================

print("🔍 MISSING VALUES (BEFORE CLEANING)")
print("=" * 50)

datasets = {
    "Cards":   cards_df,
    "Decks":   decks_df,
    "Battles": battles_df,
    "Meta":    meta_df
}

for name, df in datasets.items():
    total_missing = df.isnull().sum().sum()
    print(f"📦 {name:10} → Missing: {total_missing}")

🔍 MISSING VALUES (BEFORE CLEANING)
📦 Cards      → Missing: 0
📦 Decks      → Missing: 0
📦 Battles    → Missing: 0
📦 Meta       → Missing: 0


In [5]:
# ====================================================
# 🧹 CLEAN CARDS DATASET
# ====================================================

print("🧹 CLEANING CARDS DATASET")
print("=" * 50)

initial_rows = len(cards_df)

# Step 1: Remove duplicates
cards_df = cards_df.drop_duplicates(subset=['name'])
print(f"✅ Removed duplicate cards: {initial_rows - len(cards_df)}")

# Step 2: Handle missing values
cards_df = cards_df.dropna()
print(f"✅ Dropped rows with NaN values")

# Step 3: Standardize text columns
cards_df['name'] = cards_df['name'].str.strip()
cards_df['rarity'] = cards_df['rarity'].str.strip().str.title()
cards_df['type'] = cards_df['type'].str.strip().str.title()
cards_df['archetype'] = cards_df['archetype'].str.strip().str.title()
print("✅ Standardized text columns")

# Step 4: Validate elixir costs (1-9 range)
cards_df = cards_df[(cards_df['elixir_cost'] >= 1) & (cards_df['elixir_cost'] <= 9)]
print("✅ Validated elixir cost range (1-9)")

# Step 5: Validate win rates (0-100%)
cards_df = cards_df[(cards_df['win_rate'] >= 0) & (cards_df['win_rate'] <= 100)]
print("✅ Validated win rate range (0-100%)")

# Step 6: Reset index
cards_df = cards_df.reset_index(drop=True)

print(f"\n✨ Final cards count: {len(cards_df)}")
cards_df.head()

🧹 CLEANING CARDS DATASET
✅ Removed duplicate cards: 1
✅ Dropped rows with NaN values
✅ Standardized text columns
✅ Validated elixir cost range (1-9)
✅ Validated win rate range (0-100%)

✨ Final cards count: 97


,card_id,name,elixir_cost,rarity,type,arena_unlocked,damage,hitpoints,archetype,win_rate,usage_rate,popularity_score
0,1,Knight,3,Common,Troop,0,79,685,Beatdown,49.87,23.82,34.24
1,2,Archers,3,Common,Troop,0,42,125,Control,54.52,15.37,31.03
2,3,Bomber,3,Common,Troop,2,75,150,Control,47.03,4.74,21.66
3,4,Goblins,2,Common,Troop,1,47,80,Cycle,45.76,21.79,31.38
4,5,Spear Goblins,2,Common,Troop,1,32,52,Cycle,52.81,17.99,31.92


In [6]:
# ====================================================
# ⚡ FEATURE ENGINEERING - CARDS
# ====================================================

print("⚡ ADDING NEW FEATURES TO CARDS")
print("=" * 50)

# Feature 1: Elixir efficiency (damage per elixir)
cards_df['damage_per_elixir'] = round(cards_df['damage'] / cards_df['elixir_cost'], 2)

# Feature 2: HP per elixir (tankiness)
cards_df['hp_per_elixir'] = round(cards_df['hitpoints'] / cards_df['elixir_cost'], 2)

# Feature 3: Card strength score (custom metric)
cards_df['strength_score'] = round(
    (cards_df['win_rate'] * 0.5) +
    (cards_df['usage_rate'] * 0.3) +
    (cards_df['popularity_score'] * 0.2), 2
)

# Feature 4: Cost category
def cost_category(elixir):
    if elixir <= 2:
        return "Cheap"
    elif elixir <= 4:
        return "Medium"
    elif elixir <= 6:
        return "Expensive"
    else:
        return "Heavy"

cards_df['cost_category'] = cards_df['elixir_cost'].apply(cost_category)

print("✅ Added: damage_per_elixir")
print("✅ Added: hp_per_elixir")
print("✅ Added: strength_score")
print("✅ Added: cost_category")

cards_df.head()

⚡ ADDING NEW FEATURES TO CARDS
✅ Added: damage_per_elixir
✅ Added: hp_per_elixir
✅ Added: strength_score
✅ Added: cost_category


,card_id,name,elixir_cost,rarity,type,arena_unlocked,damage,hitpoints,archetype,win_rate,usage_rate,popularity_score,damage_per_elixir,hp_per_elixir,strength_score,cost_category
0,1,Knight,3,Common,Troop,0,79,685,Beatdown,49.87,23.82,34.24,26.33,228.33,38.93,Medium
1,2,Archers,3,Common,Troop,0,42,125,Control,54.52,15.37,31.03,14.00,41.67,38.08,Medium
2,3,Bomber,3,Common,Troop,2,75,150,Control,47.03,4.74,21.66,25.00,50.00,29.27,Medium
3,4,Goblins,2,Common,Troop,1,47,80,Cycle,45.76,21.79,31.38,23.50,40.00,35.69,Cheap
4,5,Spear Goblins,2,Common,Troop,1,32,52,Cycle,52.81,17.99,31.92,16.00,26.00,38.19,Cheap


In [7]:
# ====================================================
# 🧹 CLEAN DECKS DATASET
# ====================================================

print("🧹 CLEANING DECKS DATASET")
print("=" * 50)

initial_rows = len(decks_df)

# Step 1: Remove duplicates
decks_df = decks_df.drop_duplicates()
print(f"✅ Removed duplicate decks: {initial_rows - len(decks_df)}")

# Step 2: Drop rows with missing values
decks_df = decks_df.dropna()

# Step 3: Validate elixir range
decks_df = decks_df[(decks_df['avg_elixir'] >= 1) & (decks_df['avg_elixir'] <= 9)]
print("✅ Validated avg elixir range")

# Step 4: Validate win rates
decks_df = decks_df[(decks_df['win_rate'] >= 0) & (decks_df['win_rate'] <= 100)]
print("✅ Validated win rate range")

# Step 5: Add deck quality category
def deck_quality(win_rate):
    if win_rate >= 60:
        return "Excellent"
    elif win_rate >= 50:
        return "Good"
    elif win_rate >= 45:
        return "Average"
    else:
        return "Weak"

decks_df['quality'] = decks_df['win_rate'].apply(deck_quality)
print("✅ Added deck quality category")

# Step 6: Reset index
decks_df = decks_df.reset_index(drop=True)

print(f"\n✨ Final decks count: {len(decks_df)}")
decks_df.head()

🧹 CLEANING DECKS DATASET
✅ Removed duplicate decks: 0
✅ Validated avg elixir range
✅ Validated win rate range
✅ Added deck quality category

✨ Final decks count: 500


,deck_id,card_1,card_2,card_3,card_4,card_5,card_6,card_7,card_8,avg_elixir,archetype,win_rate,usage_count,total_battles,quality
0,1,Goblin Barrel,Tornado,Royal Recruits,PEKKA,Furnace,Mortar,X-Bow,Barbarians,4.88,Beatdown,53.82,5901,20190,Good
1,2,Zappies,Royal Delivery,Lightning,Cannon,Cannon Cart,Balloon,Phoenix,Firecracker,4.12,Beatdown,52.29,9927,38220,Good
2,3,Fisherman,Baby Dragon,Mother Witch,Executioner,Skeletons,Bandit,Royal Hogs,Ice Spirit,3.25,Cycle,56.81,7751,2167,Good
3,4,Inferno Tower,Tesla,Flying Machine,Sparky,Monk,Archers,Goblinstein,Ram Rider,4.62,Control,44.24,6919,37487,Weak
4,5,Firecracker,Prince,Goblin Cage,Arrows,Witch,Magic Archer,Hunter,Sparky,4.25,Control,55.43,6981,7949,Good


In [8]:
# ====================================================
# 🧹 CLEAN BATTLES DATASET
# ====================================================

print("🧹 CLEANING BATTLES DATASET")
print("=" * 50)

initial_rows = len(battles_df)

# Step 1: Remove duplicates
battles_df = battles_df.drop_duplicates()
print(f"✅ Removed duplicate battles: {initial_rows - len(battles_df)}")

# Step 2: Drop missing values
battles_df = battles_df.dropna()

# Step 3: Validate trophy range (0 - 10,000)
battles_df = battles_df[
    (battles_df['player_1_trophies'] >= 0) & (battles_df['player_1_trophies'] <= 10000) &
    (battles_df['player_2_trophies'] >= 0) & (battles_df['player_2_trophies'] <= 10000)
]
print("✅ Validated trophy ranges")

# Step 4: Validate crown count (0-3)
battles_df = battles_df[
    (battles_df['player_1_crowns'].between(0, 3)) &
    (battles_df['player_2_crowns'].between(0, 3))
]
print("✅ Validated crown counts (0-3)")

# Step 5: Add new feature - trophy difference
battles_df['trophy_difference'] = abs(
    battles_df['player_1_trophies'] - battles_df['player_2_trophies']
)

# Step 6: Battle intensity (sum of crowns)
battles_df['battle_intensity'] = (
    battles_df['player_1_crowns'] + battles_df['player_2_crowns']
)

# Step 7: Match type
def match_type(diff):
    if diff < 100:
        return "Balanced"
    elif diff < 500:
        return "Slight Mismatch"
    else:
        return "Heavy Mismatch"

battles_df['match_type'] = battles_df['trophy_difference'].apply(match_type)
print("✅ Added: trophy_difference, battle_intensity, match_type")

# Reset index
battles_df = battles_df.reset_index(drop=True)

print(f"\n✨ Final battles count: {len(battles_df)}")
battles_df.head()

🧹 CLEANING BATTLES DATASET
✅ Removed duplicate battles: 0
✅ Validated trophy ranges
✅ Validated crown counts (0-3)
✅ Added: trophy_difference, battle_intensity, match_type

✨ Final battles count: 5000


,battle_id,player_1_deck,player_1_elixir,player_1_trophies,player_1_crowns,player_2_deck,player_2_elixir,player_2_trophies,player_2_crowns,winner,battle_duration_sec,game_mode,trophy_difference,battle_intensity,match_type
0,1,Mini PEKKA|Fire Spirit|Royal Champion|Valkyrie...,4.00,2453,2,Lumberjack|Balloon|Goblin Gang|Ice Golem|Hunte...,3.88,6238,3,player_2,115,Tournament,3785,5,Heavy Mismatch
1,2,PEKKA|Fisherman|Cannon Cart|Royal Champion|Min...,5.50,2413,3,Fireball|Battle Healer|Skeleton Barrel|Earthqu...,3.25,6291,0,player_1,285,Ladder,3878,3,Heavy Mismatch
2,3,Electro Wizard|Mortar|Freeze|Royal Recruits|Ba...,4.62,5995,3,Archers|Battle Healer|Lava Hound|Archer Queen|...,4.75,3001,1,player_1,124,Challenge,2994,4,Heavy Mismatch
3,4,X-Bow|Musketeer|Elixir Golem|Bowler|Freeze|Roy...,4.38,3461,1,Archers|Miner|Mortar|Flying Machine|Wall Break...,3.44,6225,2,player_2,224,Challenge,2764,3,Heavy Mismatch
4,5,Hog Rider|Giant|Minions|Baby Dragon|Furnace|Go...,4.33,6879,0,Balloon|Ice Wizard|Skeleton Army|Witch|Magic A...,3.38,5152,1,player_2,269,Ladder,1727,1,Heavy Mismatch


In [9]:
# ====================================================
# 🧹 CLEAN META STATS DATASET
# ====================================================

print("🧹 CLEANING META DATASET")
print("=" * 50)

initial_rows = len(meta_df)

# Remove duplicates
meta_df = meta_df.drop_duplicates()
print(f"✅ Removed duplicates: {initial_rows - len(meta_df)}")

# Drop NaN
meta_df = meta_df.dropna()

# Standardize text
meta_df['card_name'] = meta_df['card_name'].str.strip()
meta_df['season'] = meta_df['season'].str.strip()
meta_df['trend'] = meta_df['trend'].str.strip().str.title()

# Validate
meta_df = meta_df[(meta_df['win_rate'] >= 0) & (meta_df['win_rate'] <= 100)]
meta_df = meta_df[(meta_df['usage_rate'] >= 0) & (meta_df['usage_rate'] <= 100)]

# Reset index
meta_df = meta_df.reset_index(drop=True)

print(f"\n✨ Final meta records: {len(meta_df)}")
meta_df.head()

🧹 CLEANING META DATASET
✅ Removed duplicates: 0

✨ Final meta records: 490


,card_name,season,usage_rate,win_rate,ranking,trend
0,Knight,Season_45,23.52,50.59,50,Falling
1,Knight,Season_46,24.64,57.83,68,Falling
2,Knight,Season_47,2.69,43.74,99,Rising
3,Knight,Season_48,22.68,47.00,82,Falling
4,Knight,Season_49,14.09,57.58,38,Falling


In [10]:
# ====================================================
# ✅ VERIFY CLEANING RESULTS
# ====================================================

print("✅ MISSING VALUES (AFTER CLEANING)")
print("=" * 50)

cleaned_datasets = {
    "Cards":   cards_df,
    "Decks":   decks_df,
    "Battles": battles_df,
    "Meta":    meta_df
}

for name, df in cleaned_datasets.items():
    missing = df.isnull().sum().sum()
    duplicates = df.duplicated().sum()
    print(f"📦 {name:10} → Missing: {missing} | Duplicates: {duplicates} | Rows: {len(df)}")

✅ MISSING VALUES (AFTER CLEANING)
📦 Cards      → Missing: 0 | Duplicates: 0 | Rows: 97
📦 Decks      → Missing: 0 | Duplicates: 0 | Rows: 500
📦 Battles    → Missing: 0 | Duplicates: 0 | Rows: 5000
📦 Meta       → Missing: 0 | Duplicates: 0 | Rows: 490


In [11]:
# ====================================================
# 💾 SAVE CLEANED DATASETS
# ====================================================

print("💾 SAVING CLEANED DATASETS")
print("=" * 50)

cards_df.to_csv(PROCESSED_PATH / "cards_cleaned.csv", index=False)
print("✅ Saved: cards_cleaned.csv")

decks_df.to_csv(PROCESSED_PATH / "decks_cleaned.csv", index=False)
print("✅ Saved: decks_cleaned.csv")

battles_df.to_csv(PROCESSED_PATH / "battles_cleaned.csv", index=False)
print("✅ Saved: battles_cleaned.csv")

meta_df.to_csv(PROCESSED_PATH / "meta_cleaned.csv", index=False)
print("✅ Saved: meta_cleaned.csv")

print(f"\n📂 All cleaned files saved in: {PROCESSED_PATH.absolute()}")

💾 SAVING CLEANED DATASETS
✅ Saved: cards_cleaned.csv
✅ Saved: decks_cleaned.csv
✅ Saved: battles_cleaned.csv
✅ Saved: meta_cleaned.csv

📂 All cleaned files saved in: e:\Clash Royale\notebooks\..\dataset\processed


In [12]:
# ====================================================
# 📝 CLEANING SUMMARY REPORT
# ====================================================

print("=" * 70)
print("📝 DATA CLEANING SUMMARY REPORT")
print("=" * 70)

print(f"""
🃏 CARDS DATASET
  ✅ Total cards:      {len(cards_df)}
  ✅ New features:     damage_per_elixir, hp_per_elixir, strength_score, cost_category
  ✅ Rarities:         {cards_df['rarity'].nunique()}
  ✅ Cost categories:  {cards_df['cost_category'].nunique()}

🎴 DECKS DATASET
  ✅ Total decks:      {len(decks_df)}
  ✅ New features:     quality
  ✅ Excellent decks:  {(decks_df['quality'] == 'Excellent').sum()}
  ✅ Avg win rate:     {decks_df['win_rate'].mean():.2f}%

⚔️  BATTLES DATASET
  ✅ Total battles:    {len(battles_df)}
  ✅ New features:     trophy_difference, battle_intensity, match_type
  ✅ Balanced matches: {(battles_df['match_type'] == 'Balanced').sum()}

📊 META DATASET
  ✅ Total records:    {len(meta_df)}
  ✅ Seasons covered:  {meta_df['season'].nunique()}

🚀 NEXT STEP: Notebook 03 - Exploratory Data Analysis (EDA) with Visualizations!
""")

📝 DATA CLEANING SUMMARY REPORT

🃏 CARDS DATASET
  ✅ Total cards:      97
  ✅ New features:     damage_per_elixir, hp_per_elixir, strength_score, cost_category
  ✅ Rarities:         5
  ✅ Cost categories:  4

🎴 DECKS DATASET
  ✅ Total decks:      500
  ✅ New features:     quality
  ✅ Excellent decks:  92
  ✅ Avg win rate:     52.46%

⚔️  BATTLES DATASET
  ✅ Total battles:    5000
  ✅ New features:     trophy_difference, battle_intensity, match_type
  ✅ Balanced matches: 193

📊 META DATASET
  ✅ Total records:    490
  ✅ Seasons covered:  5

🚀 NEXT STEP: Notebook 03 - Exploratory Data Analysis (EDA) with Visualizations!

